# Adaptation Hypothesis Demo (0004)

Thin notebook -- logic lives in `data/*.py` and `probe/*.py`. Follows `docs/decisions/0004-adaptation-hypothesis-demo-build.md` and `docs/notes/2026-08-02-build-plan.md` in stage order (S1-S10).

**Rule with no exceptions** (build plan S0): after every stage that touches data, open the images and look at them.

In [ ]:
import os
if os.path.isdir("deepfake"):
    %cd deepfake
    !git pull
else:
    !git clone https://github.com/satyagalla/deepfake.git
    %cd deepfake

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
# A100 expected per build plan Environment section.

In [ ]:
# Mount Drive for raw/checkpoint/feature persistence across runtime recycles.
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['DEEPFAKE_DATA_ROOT'] = '/content/drive/MyDrive/deepfake'

## S1 -- Data (~45 min, highest risk, do first)

Fixes `0003`'s single highest-value finding: `data/download.py:68` discarded 5 of COCO_AI's 6 generator columns. Now takes all 7 (real + sd21/sdxl/sd3/sd35/dalle/midjourney), drops the person-caption filter (existed only for the face crop `0003` §4.1 already removed), and writes complete 7-way rows only -- a partial row would silently break the 1:1 pairing invariant (I2/I3).

In [ ]:
from data.download import test_sources

# No-download validation: HF schema (all 7 columns). skip_casia=True -- CASIA is the
# older face-filtered v1 pipeline's source (needs Kaggle credentials this notebook
# doesn't set up) and probe/ never reads it.
test_sources(skip_casia=True)

In [ ]:
from data.download import download_all

# Full download. n_pairs=3000 is the build-plan default; fall back to 1000 if this
# isn't done by the 60-minute abort condition (build plan S1) -- the N-shot claim
# does not need 3000 rows. skip_casia=True -- probe/ never reads CASIA (see above).
download_all(n_pairs=3000, skip_casia=True)

In [ ]:
# CHECKPOINT (build plan S1, no exceptions): open a few images per column and look
# at them. Confirm the real/fake pairing is intact for a sampled row.
from PIL import Image
from config import COCO_AI_RAW_DIR, COCO_AI_COLUMNS
import matplotlib.pyplot as plt

row_id = '00000'
fig, axes = plt.subplots(1, len(COCO_AI_COLUMNS), figsize=(3 * len(COCO_AI_COLUMNS), 3))
for ax, key in zip(axes, COCO_AI_COLUMNS):
    img_path = COCO_AI_RAW_DIR / key / f'{row_id}.jpg'
    ax.imshow(Image.open(img_path))
    ax.set_title(key, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Native resolution histogram per column (build plan S1 step 4) -- determines
# whether N=16 patches (0004 I1) is the right count.
!python data/resolution_stats.py

## S2 -- Self-generated N-shot pool (~45 min, manual generation step)

Generation itself is manual: Gemini + GPT-Image, COCO-caption prompted (content-domain matched) and off-domain prompted (E4), through both the API and the web UI (container control -- SynthID passes the naive container control by design, `bottlenecks.md` §4.2). This notebook only produces the prompts and organizes whatever lands in the intake folder.

In [ ]:
# Prints ~130 in-domain (COCO caption) + ~20-30 off-domain prompts to paste into
# Gemini/GPT-Image, both via the API and the web UI.
!python data/selfgen_prompts.py --indomain 130 --offdomain 20 --out selfgen_prompts.txt
print(open('selfgen_prompts.txt').read())

**Manual step:** generate images from the prompts above and drop them into
```
data_raw/selfgen_intake/<gemini|gptimage>/<indomain|offdomain>/<api|web>/*
```
e.g. `data_raw/selfgen_intake/gemini/indomain/web/img003.webp`. ~150-300 images total (build plan S2), ~20/generator off-domain, ~20/generator via the web UI. Then run the organizer cell below.

In [ ]:
# Re-saves everything at JPEG q95 (I4), builds the manifest, and prints the
# container-control check (does web UI actually differ from API in file format).
!python data/selfgen_organize.py

In [ ]:
# CHECKPOINT: look at a few organized self-gen images per generator/domain.
from config import SELFGEN_RAW_DIR, SELFGEN_GENERATORS
import itertools

samples = []
for gen in SELFGEN_GENERATORS:
    for p in sorted((SELFGEN_RAW_DIR / gen).rglob('*.jpg'))[:2]:
        samples.append(p)
fig, axes = plt.subplots(1, len(samples), figsize=(3 * len(samples), 3))
for ax, p in zip(axes, samples):
    ax.imshow(Image.open(p))
    ax.set_title(p.relative_to(SELFGEN_RAW_DIR).as_posix(), fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Row-level split (I2/I3/I7)

Splits COCO_AI at the row level (a row's real + all 6 fakes stay together), holds Midjourney out entirely (I7 -- never trained on at any N), and partitions the self-gen pool into an N-shot adaptation pool + a fixed eval slice per generator/domain.

In [ ]:
!python -m probe.split

## S3 -- Feature extraction (~45 min)

Frozen CLIP ViT-L/14 @224, fp16. N=16 native-resolution patches (I1, no resampling) + 1 whole-image resized view, CLIP's own normalization (I5), L2-normalized (I6). Second arm (`--arm standard`) is the ordinary resize+centre-crop pipeline -- UniversalFakeDetect's exact preprocessing, the reproducible control for E7.

In [ ]:
!python -m probe.extract
!python -m probe.extract --arm standard

In [ ]:
# CHECKPOINT: I1 assertion already runs inside extraction; spot-check feature norms are 1.0 (I6).
import numpy as np
from config import FEATURES_DIR
from probe.features import read_index

rows = read_index(FEATURES_DIR)[:5]
for r in rows:
    d = np.load(r.feature_path)
    print(r.key, 'patches:', d['patches'].shape, 'whole norm:', np.linalg.norm(d['whole']))

## S4 -- Heads (~30 min)

Head A (binary, **no calibration stage** -- `0005` §3), Head B (generator ID), Mahalanobis OOD gate (pooled covariance, Ledoit-Wolf shrinkage), and the Spectral card's head -- all fit in seconds off the cached features. Saved to `config.PROBE_CHECKPOINT_DIR` for `probe/demo.py`.

In [ ]:
!python -m probe.fit_production

## S5 -- E1, the headline

N-shot adaptation curve: accuracy on gemini/gptimage vs N images from it, N in {0,5,10,20,30,50,100}, multiple random draws per N. **The knee is the finding, wherever it is** -- a curve with no knee falsifies the claim (0004 §9).

## S6 -- free experiments E2/E5/E6/E7, and E4

All come off the S3 cache at zero marginal GPU cost: E2 (Midjourney 0-shot), E5 (AUC, first measurement on this project), E6 (aggregator ablation), E7 (native patches vs standard resize), E4 (off-domain vs in-domain accuracy).

In [ ]:
!python -m probe.experiments

In [ ]:
# E1 curves, rendered to PROBE_OUTPUTS_DIR during the run above.
from config import PROBE_OUTPUTS_DIR, SELFGEN_GENERATORS
for gen in SELFGEN_GENERATORS:
    p = PROBE_OUTPUTS_DIR / f'e1_{gen}.png'
    if p.exists():
        display(Image.open(p))

## S9 -- E3, degradation ladder

Score each card across re-encode / rescale / re-render. A learned score (AI Model) that survives degradation identically to the provenance card (EXIF, which is stripped by any re-encode) is reading a watermark rather than a synthesis artifact. Gemini vs GPT-Image differ in watermarking policy -- the natural experiment.

Lowest priority in the cut order (build plan §3) -- skip this cell if time is short.

In [ ]:
!python -m probe.degradation
from config import PROBE_OUTPUTS_DIR
display(Image.open(PROBE_OUTPUTS_DIR / 'e3_degradation.png'))

## S7 -- Evidence cards + fusion

Built inside `probe/cards.py` and exercised live by the demo below (also P1 drill #1 -- the drill implements **Plurall's** spec; this card layer uses **our** derived vocabulary, `0005` §12). Quick standalone check on one held-out image:

In [ ]:
from pathlib import Path
from probe.extract import build_clip, featurize_single_image
from probe.cards import run_all_cards
from probe.heads import load
from config import HEAD_A_NAME, HEAD_B_NAME, GATE_NAME, SPECTRAL_HEAD_NAME, SELFGEN_RAW_DIR, SELFGEN_GENERATORS

model, preprocess_val = build_clip()
head_a, head_b, gate = load(HEAD_A_NAME), load(HEAD_B_NAME), load(GATE_NAME)
spectral_fitted = load(SPECTRAL_HEAD_NAME)

sample_path = next((SELFGEN_RAW_DIR / SELFGEN_GENERATORS[0]).rglob('*.jpg'))
sample_image = Image.open(sample_path).convert('RGB')
x_row = featurize_single_image(sample_image, model, preprocess_val)
cards, fusion = run_all_cards(sample_image, sample_path, x_row, head_a, head_b, gate, spectral_fitted)
for c in cards:
    print(f'{c.dimension:16s} score={c.score} silent={c.silent_because} {c.label}')
print('FUSION:', fusion.verdict, fusion.fused_score, 'reliability=', fusion.reliability)

## S8 -- Gradio demo

New path: no face crop, no MTCNN. Displays the fused score, verdict, all six cards, the abstention state, and E1's curve on screen with the live upload's score marked on it.

In [ ]:
from probe.demo import build_app

demo = build_app()
demo.launch(share=True)  # share=True for the live interview demo